In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

RANDOM_STATE = 42

In [ ]:

DATA_PATH = Path('ready_for_featrue_selection.csv')
if not DATA_PATH.exists():
    DATA_PATH = Path('C:\\Users\\koemhort.leng\\Desktop\\EWS_NPL_V2\\EDA\\ready_for_featrue_selection.csv')

df = pd.read_csv(DATA_PATH)
print('Loaded:', DATA_PATH)
print('Shape:', df.shape)
display(df.head())

In [ ]:
df.loc[
    df["ACCOUNT_OFFICER"].notnull(),
    ["ACCOUNT_OFFICER"]
].head(5) 

## Check the target

`IS_NPL` must be coded as 0/1 before any selection method below can use it.

In [ ]:
TARGET = 'IS_NPL'
print(df[TARGET].value_counts(dropna=False))
print((df[TARGET].value_counts(normalize=True) * 100).round(2))

if not set(df[TARGET].dropna().unique()).issubset({0, 1}):
    raise ValueError('IS_NPL must be coded as 0 and 1 before training.')

## Exclude IDs and leakage columns

IDs and post-outcome fields are never candidates for selection; they are dropped before any ranking method
sees the data

In [ ]:
# IDs -- no predictive meaning; would leak row/account identity, not risk signal.
ID_COLS = {'CUSTOMER_ID', 'ACCOUNT_NUM', 'TEL_MOBILE', 'BRANCH_CODE', 'BRANCH_CODE_1'}

# Raw datetime fields already excluded via DOB/APP_DATE/OPENING_DATE/MATURITY_DATE below.
# REPAY_ST_DATE scores IV = 1.74 (way above the 0.5 "suspicious" threshold) -- exclude until it is
# re-engineered into tenure/age-style numeric features rather than used as a raw date/category.
DATE_LEAKAGE_COLS = {'REPAY_ST_DATE'}

# Confirmed target leakage: IS_NPL rate by M0_PSC is exactly 100% for buckets 4-13 and exactly 0% for
# buckets 0-1 -- these scores define/derive the target rather than predict it independently. IV for
# M0-M12_PSC ranges 4.07-12.39, far above the 0.5 "suspicious" threshold.
PSC_LEAKAGE_COLS = {f'M{i}_PSC' for i in range(13)}

# Correlated with IS_NPL (15.3% vs 8.6% and 17.7% vs 9.1% NPL rate) but timing relative to the NPL
# outcome is unconfirmed -- treat as leakage until confirmed with whoever defines the observation window.
UNCONFIRMED_TIMING_COLS = {'PMS_PASSDUE', 'PMS_RESTRUCTURE'}

EXCLUDE = {
    TARGET, 'LOAN_STATUS', 'TARGET_DESC', 'BUSINESS_DATE',
    'DOB', 'APP_DATE', 'OPENING_DATE', 'MATURITY_DATE',
} | ID_COLS | DATE_LEAKAGE_COLS | PSC_LEAKAGE_COLS | UNCONFIRMED_TIMING_COLS

EXCLUDE = {c for c in EXCLUDE if c in df.columns}
candidates = [c for c in df.columns if c not in EXCLUDE]
print('Excluded:', sorted(EXCLUDE))
print('Candidate count:', len(candidates))

In [ ]:
print(candidates)

In [ ]:
print(len(candidates), 'candidate features for IV/selection')

## Filter: data quality

Drop features that are almost entirely missing (>95%) or have a single value (constant). Neither can teach
a model anything.

In [ ]:
quality_rows = []
usable = []
for col in candidates:
    missing_pct = df[col].isna().mean() * 100
    unique_count = df[col].nunique(dropna=True)
    reason = ''
    if missing_pct > 95:
        reason = 'over_95pct_missing'
    elif unique_count <= 1:
        reason = 'constant'
    else:
        usable.append(col)
    quality_rows.append({'feature': col, 'missing_pct': round(missing_pct, 2), 'unique_count': unique_count, 'drop_reason': reason})

quality_report = pd.DataFrame(quality_rows)

dropped = quality_report[quality_report['drop_reason'] != '']
print(f"Dropped ({len(dropped)} of {len(quality_report)} candidates):")
display(dropped)

print('\nFull quality report (all candidates):')
display(quality_report.sort_values(['drop_reason', 'missing_pct'], ascending=[False, False]))
print('\nUsable after quality filter:', len(usable))

## Filter: Information Value (IV) and Weight of Evidence (WOE)

IV is the standard credit-scorecard measure of how strongly a feature separates good accounts from NPL
accounts, independent of the model used later. Numeric features are binned into deciles; categorical
features use their own categories as bins.

Rule of thumb: IV < 0.02 not predictive, 0.02–0.1 weak, 0.1–0.3 medium, 0.3–0.5 strong,
> 0.5 suspiciously strong — check for leakage before trusting it.

In [ ]:
def calculate_iv(feature_series, target_series, max_bins=10):
    x = feature_series
    if pd.api.types.is_numeric_dtype(x) and x.nunique(dropna=True) > max_bins:
        ranks = x.rank(method='first')
        bucket = pd.qcut(ranks, q=max_bins, duplicates='drop').astype('string')
    else:
        bucket = x.astype('string')
    bucket = bucket.fillna('Missing')

    table = pd.DataFrame({'bucket': bucket, 'target': target_series})
    grouped = table.groupby('bucket')['target'].agg(total='count', bad='sum')
    grouped['good'] = grouped['total'] - grouped['bad']
    total_bad = grouped['bad'].sum()
    total_good = grouped['good'].sum()
    dist_bad = (grouped['bad'] + 0.5) / (total_bad + 0.5 * len(grouped))
    dist_good = (grouped['good'] + 0.5) / (total_good + 0.5 * len(grouped))
    woe = np.log(dist_good / dist_bad)
    return ((dist_good - dist_bad) * woe).sum()


def iv_strength(iv):
    if iv < 0.02:
        return 'useless'
    if iv < 0.1:
        return 'weak'
    if iv < 0.3:
        return 'medium'
    if iv < 0.5:
        return 'strong'
    return 'suspicious_check_leakage'


iv_rows = []
for col in usable:
    try:
        iv = calculate_iv(df[col], df[TARGET])
    except Exception:
        iv = np.nan
    iv_rows.append({'feature': col, 'iv': iv, 'iv_strength': iv_strength(iv) if pd.notna(iv) else 'error'})

iv_table = pd.DataFrame(iv_rows).sort_values('iv', ascending=False).reset_index(drop=True)
print(f'IV for all {len(iv_table)} candidates:')
display(iv_table)

suspicious = iv_table.loc[iv_table['iv_strength'] == 'suspicious_check_leakage', 'feature'].tolist()
if suspicious:
    print('Review for leakage before using (IV > 0.5):', suspicious)

useless = iv_table.loc[iv_table['iv_strength'] == 'useless', 'feature'].tolist()
usable = [c for c in usable if c not in useless]
print('Usable after IV filter (dropped IV < 0.02):', len(usable))

In [ ]:
print(len(useless))

In [ ]:
print(useless)

In [ ]:
print(len(usable))

In [ ]:
print(usable)

In [ ]:
iv_table_usable = iv_table[iv_table['feature'].isin(usable)].reset_index(drop=True)

print(f'IV for usable {len(iv_table_usable)} candidates:')
display(iv_table_usable)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

numeric_cols = [
    c for c in usable
    if pd.api.types.is_numeric_dtype(df[c])
]

corr = df[numeric_cols].corr()

plt.figure(figsize=(14, 10))

sns.heatmap(
    corr,
    cmap='coolwarm',
    center=0,
    vmin=-1,
    vmax=1
)

plt.title('Correlation Heatmap of Numeric Features')
plt.tight_layout()
plt.show()

In [ ]:
# Select numeric variables
numeric_cols = [
    c for c in usable
    if pd.api.types.is_numeric_dtype(df[c])
]

# Calculate correlation
corr = df[numeric_cols].corr()

# Round to 3 decimal places
corr_table = corr.round(3)

# Display full correlation matrix
display(corr_table)

## Save the final feature list for `xgboot.ipynb`

In [ ]:
OUTPUT_DIR = Path('feature_selection_output')
OUTPUT_DIR.mkdir(exist_ok=True)

selected_data = df[usable + [TARGET]].copy()
save_path = OUTPUT_DIR / 'selected_features_xgboost.csv'
selected_data.to_csv(save_path, index=False)

print('Saved:', save_path.resolve())

## Clean the Selected Feature Store (dedup + outlier treatment)

Folded in from the former separate `remove_outlier_feature_store.ipynb` notebook, so the
selected-feature cleaning happens right after selection instead of in its own file. Operates
on `selected_data` (already the same `usable + [TARGET]` columns just saved above) — drops
exact-duplicate rows, then removes IQR-based outliers, and saves the result as
`selected_features_xgboost_cleaned.csv`, which `cross_validate.ipynb` reads to build the
train/valid/test splits.

In [ ]:
# ------------------------------------------------------------------
# DEDUPLICATE
# ------------------------------------------------------------------
clean_data = selected_data.copy()
n_dupes = clean_data.duplicated().sum()
print(f"Full-row duplicates: {n_dupes} ({n_dupes / len(clean_data):.2%} of rows)")

clean_data = clean_data.drop_duplicates().reset_index(drop=True)
print('Shape after dropping duplicates:', clean_data.shape)


In [ ]:
# ------------------------------------------------------------------
# OUTLIER BOUNDS (IQR rule) -- skip zero-inflated columns where IQR == 0
# ------------------------------------------------------------------
# bool is excluded even though pd.api.types.is_numeric_dtype() treats it as
# numeric -- a boolean flag has no "outliers" in the IQR sense, and newer
# numpy raises TypeError trying to subtract booleans during quantile interpolation.
numeric_cols_clean = [
    c for c in clean_data.columns
    if c != TARGET and pd.api.types.is_numeric_dtype(clean_data[c]) and clean_data[c].dtype != bool
]

outlier_cols = []
bounds = {}
for col in numeric_cols_clean:
    q1, q3 = clean_data[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    if iqr > 0:
        lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        bounds[col] = (lower, upper)
        outlier_cols.append(col)

skipped_cols = [c for c in numeric_cols_clean if c not in outlier_cols]
print(f'Columns used for IQR outlier removal ({len(outlier_cols)}): {outlier_cols}')
print(f'Columns skipped, IQR == 0 / zero-inflated ({len(skipped_cols)}): {skipped_cols}')

impact = []
for col, (lower, upper) in bounds.items():
    n_out = (~clean_data[col].between(lower, upper)).sum()
    impact.append({'column': col, 'lower_bound': lower, 'upper_bound': upper,
                    'n_outliers': n_out, 'pct_outliers': round(n_out / len(clean_data) * 100, 2)})

impact_df = pd.DataFrame(impact).sort_values('pct_outliers', ascending=False)
display(impact_df)

combined_mask = pd.Series(True, index=clean_data.index)
for col, (lower, upper) in bounds.items():
    combined_mask &= clean_data[col].between(lower, upper)

print(f"Rows kept if filtering on ALL {len(outlier_cols)} columns at once: "
      f"{combined_mask.sum()} / {len(clean_data)} ({combined_mask.sum() / len(clean_data):.2%})")


In [ ]:
# ------------------------------------------------------------------
# APPLY OUTLIER REMOVAL (drop rows flagged on any IQR-bounded column)
# ------------------------------------------------------------------
print('Shape before outlier removal:', clean_data.shape)
clean_data = clean_data[combined_mask].reset_index(drop=True)
print('Shape after outlier removal:', clean_data.shape)
print('IS_NPL rate after cleaning:', f"{clean_data[TARGET].mean():.4%}")


In [ ]:
# ------------------------------------------------------------------
# SAVE + VERIFY THE CLEANED FEATURE STORE
# ------------------------------------------------------------------
CLEANED_PATH = OUTPUT_DIR / 'selected_features_xgboost_cleaned.csv'
clean_data.to_csv(CLEANED_PATH, index=False)
print('Saved cleaned feature store to:', CLEANED_PATH.resolve())

check = pd.read_csv(CLEANED_PATH)
print('Reloaded shape:', check.shape)
assert check.shape == clean_data.shape, "Saved file shape does not match clean_data!"
display(check.head())
